## Assessment 1

Name: Iliana Peters

Student Number: 35723483


### Dataset introduction

The dataset used in this assignment is the Uber Data Analytics Dashboard for the entire 2024 year. It includes the data of various Uber rides, from users and drivers, including dates, locations, timing data, prices, and ratings. A main business case that can be made from this data relates to influential factors in patient satisfaction, leading to the question 'What are the most influential factors that impact rider satisfaction?' Various metrics could be investigated such as as ride distance to booking value, pickup duration, vehicle type, and travel duration to ride distance. Additionally, Customer Cancellation Reasons is a free text column that can be textually analysed for trends and additional insights. 

### Part A: Analytical Query Design and Implementation
Part A assesses your ability to design, implement, and evaluate a non-trivial analytical query using Apache Spark. The focus is not only on producing correct results, but also on demonstrating an undersatnding of how Spark processes distributed workloads. 


#### 1. The Business Query
Design and implement a non-trivial business query that requires the following operations:
- A window function, for example, running total, rank within a partition, moving average, cumulative statistics
- High-velocity activity spikes: identify users whose transactions count in any single hour exceeds three standard deviations above their personal hourly mean
- Identification of geographic regions with ususually high activity compared to historical averages.

Using the above operations and relating them to the Uber dataset, the following queries are created and will be investigated in the following sections:
1. Perform a window function to determine cumulative statistics on the Ride Distance, Booking Value, Pickup Duration, Travel Duration and Customer Rating. These should be ranked by Customer Ratings within the partition. 
2. High-velocity activity spikes: identifyig users who has multiple Booking IDs within a single hour that exceeds three standard deviations above the standard user hourly mean. Correlate these to Drive and Customer Ratings.
3. Identify geographic regions with high activity compared to historical averages. From top geographical, identify Booking Dates and Booking Times that are also above the historical average.

#### 2. DataFrame Implementation

Implement the query using the Spark DataFrame API

In [40]:
from pyspark import SparkConf
from pyspark.sql import SparkSession

#setup a spark session and load dataset
master = "local[*]"
app_name = "Uber Business Query DF"
spark_conf = SparkConf().setMaster(master).setAppName(app_name)

spark = SparkSession.builder.config(conf=spark_conf).getOrCreate()
sc = spark.sparkContext
sc.setLogLevel('ERROR')

df = spark.read.csv("ncr_ride_bookings.csv",header=True)
df.show(5)
df.printSchema()
df.count()

+----------+--------+----------------+---------------+----------------+-------------+-------------------+-----------------+--------+--------+---------------------------+---------------------------------+-------------------------+--------------------------+----------------+-----------------------+-------------+-------------+--------------+---------------+--------------+
|      Date|    Time|      Booking ID| Booking Status|     Customer ID| Vehicle Type|    Pickup Location|    Drop Location|Avg VTAT|Avg CTAT|Cancelled Rides by Customer|Reason for cancelling by Customer|Cancelled Rides by Driver|Driver Cancellation Reason|Incomplete Rides|Incomplete Rides Reason|Booking Value|Ride Distance|Driver Ratings|Customer Rating|Payment Method|
+----------+--------+----------------+---------------+----------------+-------------+-------------------+-----------------+--------+--------+---------------------------+---------------------------------+-------------------------+--------------------------+

150000

In [50]:
#convert numerical columns to float from string
from pyspark.sql.functions import col
from pyspark.sql.types import DateType

# Replace string 'null' with true None/null globally
df = df.replace("null", None)

float_col = ["Avg VTAT","Avg CTAT","Ride Distance","Driver Ratings","Customer Ratings","Booking Value"]
df = df.select([col(c).cast("float") if c in float_col else col(c) for c in df.columns])
df = df.withColumn("Date", col("Date").cast(DateType()))
df = df.withColumn("Time", col("Time").cast("timestamp"))
df.printSchema()


root
 |-- Date: date (nullable = true)
 |-- Time: timestamp (nullable = true)
 |-- Booking ID: string (nullable = true)
 |-- Booking Status: string (nullable = true)
 |-- Customer ID: string (nullable = true)
 |-- Vehicle Type: string (nullable = true)
 |-- Pickup Location: string (nullable = true)
 |-- Drop Location: string (nullable = true)
 |-- Avg VTAT: float (nullable = true)
 |-- Avg CTAT: float (nullable = true)
 |-- Cancelled Rides by Customer: string (nullable = true)
 |-- Reason for cancelling by Customer: string (nullable = true)
 |-- Cancelled Rides by Driver: string (nullable = true)
 |-- Driver Cancellation Reason: string (nullable = true)
 |-- Incomplete Rides: string (nullable = true)
 |-- Incomplete Rides Reason: string (nullable = true)
 |-- Booking Value: float (nullable = true)
 |-- Ride Distance: float (nullable = true)
 |-- Driver Ratings: float (nullable = true)
 |-- Customer Rating: string (nullable = true)
 |-- Payment Method: string (nullable = true)



In [ ]:
#BUSINESS QUERY 1 - cumulative statistics on the Ride Distance, Booking Value, Pickup Duration, Travel Duration and Customer Rating.
# These should be ranked by Customer Ratings within the partition. 

print(df.count())
#filter out null ratings
filter_1 = df.dropna(subset=["Customer Rating"])
print(filter_1.count())
filter_1.describe("Customer Rating").show()


150000
93000
+-------+------------------+
|summary|   Customer Rating|
+-------+------------------+
|  count|             93000|
|   mean| 4.404583870967753|
| stddev|0.4378187328187107|
|    min|               3.0|
|    max|               5.0|
+-------+------------------+



In [52]:
#add a column that would specify the customer rating range into 0-3, 3-4, 4-5
from pyspark.sql.functions import when
filter_1 = filter_1.withColumn("Customer_Rating_Range",when(col("Customer Rating")<=3.0, "0-3").
                               when(col("Customer Rating")<=3.5, "3-3.5").when(col("Customer Rating")<=4.0, "3.5-4").
                               when(col("Customer Rating")<=4.5, "4-4.5").when(col("Customer Rating")<=5.0, "4.5-5"))
filter_1.show(20)

+----------+-------------------+----------------+--------------+----------------+-------------+-------------------+----------------+--------+--------+---------------------------+---------------------------------+-------------------------+--------------------------+----------------+-----------------------+-------------+-------------+--------------+---------------+--------------+---------------------+
|      Date|               Time|      Booking ID|Booking Status|     Customer ID| Vehicle Type|    Pickup Location|   Drop Location|Avg VTAT|Avg CTAT|Cancelled Rides by Customer|Reason for cancelling by Customer|Cancelled Rides by Driver|Driver Cancellation Reason|Incomplete Rides|Incomplete Rides Reason|Booking Value|Ride Distance|Driver Ratings|Customer Rating|Payment Method|Customer_Rating_Range|
+----------+-------------------+----------------+--------------+----------------+-------------+-------------------+----------------+--------+--------+---------------------------+----------------

In [ ]:
from pyspark.sql.window import Window

rating_window = Window.partitionBy("Customer_Rating_Range")

#Ride Distance, Booking Value, Pickup Duration, Travel Duration


#### 3. Spark SQL Implementation

Implement the same query, or a functionally equivalent version, using Spark SQL

In [ ]:
from pyspark.sql import SparkSession # Spark SQL

#### 4. Result Validaiton

Demonstrate that both implementations produce equivalent analytical results. Where minor differences occour due to sorting, formatiing, or floating-point precision, provide a brief explaination. 